In [14]:

import numpy as np
import tensorflow as tf
import os
import glob
import librosa
import math
from sklearn.model_selection import train_test_split
import sys
from io import StringIO
from collections import Counter

# Redirect stdout to capture training output and constants
old_stdout = sys.stdout
sys.stdout = StringIO()

# --- Config ---
DATASET_PATH = "recordings" 
EXPORT_NAME = "mfcc_neuron_config"
MODEL_SAVE_PATH = "mfcc_data_split.npz"

# MFCC Parameters (Fixed)
SAMPLE_RATE = 8000
FFT_LEN = 1024
HOP_LEN = 512
NUM_MEL_FILTERS = 20
NUM_DCT_COEFFS = 13
NUM_FEATURES = NUM_DCT_COEFFS

# --- C-STYLE MFCC FEATURE EXTRACTION FUNCTIONS (omitted for brevity, assume they are correct) ---
def freq_to_mel(f): return 2595.0 * np.log10(1.0 + f / 700.0)
def mel_to_freq(m): return 700.0 * (10.0**(m / 2595.0) - 1.0)
def get_filterbank(sr, n_fft, n_mels): 
    fmin_mel = freq_to_mel(20)
    fmax_mel = freq_to_mel(4000)
    mel_points = np.linspace(fmin_mel, fmax_mel, n_mels + 2)
    hz_points = mel_to_freq(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    filters = np.zeros((n_mels, n_fft // 2 + 1))
    for i in range(n_mels):
        for j in range(bin_points[i], bin_points[i+1]):
            filters[i, j] = (j - bin_points[i]) / (bin_points[i+1] - bin_points[i])
        for j in range(bin_points[i+1], bin_points[i+2]):
            filters[i, j] = (bin_points[i+2] - j) / (bin_points[i+2] - bin_points[i+1])
    return filters

def get_dct_matrix(n_dct, n_mels): 
    matrix = np.zeros((n_dct, n_mels))
    normalizer = np.sqrt(2.0 / n_mels)
    for k in range(n_dct):
        for n in range(n_mels):
            matrix[k, n] = normalizer * np.cos(math.pi / n_mels * (n + 0.5) * k)
    return matrix

DCT_MAT = get_dct_matrix(NUM_DCT_COEFFS, NUM_MEL_FILTERS)
MEL_FILTERS = get_filterbank(SAMPLE_RATE, FFT_LEN, NUM_MEL_FILTERS)
WINDOW = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(FFT_LEN) / (FFT_LEN - 1))

def extract_mfccs(file_path):
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        audio, _ = librosa.effects.trim(audio)
        audio = audio * 32767.0 
        num_frames = (len(audio) - FFT_LEN) // HOP_LEN + 1
        if num_frames < 1: return None
        mfcc_accum = np.zeros(NUM_DCT_COEFFS)
        for i in range(num_frames):
            start = i * HOP_LEN
            frame = audio[start : start + FFT_LEN] * WINDOW
            fft_out = np.fft.rfft(frame)
            power_spec = (np.abs(fft_out) ** 2)
            mel_energies = np.dot(MEL_FILTERS, power_spec)
            log_mel = np.log(np.maximum(mel_energies, 1e-12)) 
            mfccs = np.dot(DCT_MAT, log_mel)
            mfcc_accum += mfccs
        return mfcc_accum / num_frames
    except Exception:
        return None

# --- 2. DATA PREPARATION (WITH FILTERING) ---

def prepare_data(data_dir):
    wav_files = glob.glob(os.path.join(data_dir, "*.wav"))
    
    X_raw, y = [], []
    print(f"Total WAV files found: {len(wav_files)}. Filtering for '0' and '1'...")
    
    for f in wav_files:
        try:
            # Check the prefix
            label_prefix = os.path.basename(f).split('_')[0]
            
            # --- FILTERING LOGIC ---
            if label_prefix == '0':
                label_bin = 1.0 # Target class (e.g., 'zero')
            elif label_prefix == '1':
                label_bin = 0.0 # Other class (e.g., 'one' / Not target)
            else:
                continue # Skip all files not starting with '0' or '1'
            # -----------------------

            feat = extract_mfccs(f)
            if feat is not None and feat.shape[0] == NUM_FEATURES:
                X_raw.append(feat)
                y.append(label_bin)
        except:
            continue
                
    X_raw = np.array(X_raw, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    
    print(f"Processed samples after filtering: {len(X_raw)}")
    
    return X_raw, y

# --- 3. EXPORT TO C (Assumed to be correct) ---
def export_c(model, filename, mean, std):
    w, b = model.layers[0].get_weights()
    w_flat = w.flatten()
    b_val = b[0]
    
    # ... (C file writing logic assumed) ...
    
    # Export Header
    with open(f"{filename}.h", "w") as f:
        f.write(f"#ifndef {filename.upper()}_H\n#define {filename.upper()}_H\n\n")
        f.write(f"#define NUM_FEATURES {NUM_FEATURES}\n")
        f.write("float neuron_predict(float *features);\n")
        f.write("#endif\n")
        
    # Export Source
    with open(f"{filename}.c", "w") as f:
        f.write(f'#include "{filename}.h"\n#include <math.h>\\n\\n')
        
        # Weights
        f.write(f"static const float W[{NUM_FEATURES}] = {{\n    ")
        for i, val in enumerate(w_flat):
            f.write(f"{val:.8f}f, ")
            if (i+1)%5==0 and i < len(w_flat) - 1: f.write("\n    ")
        f.write(f"\n}};")
        
        # Bias
        f.write(f"\n\nstatic const float B = {b_val:.8f}f;\n\n")
        
        # Prediction Function
        f.write("float neuron_predict(float *x) {\n")
        f.write("    float z = B;\n")
        f.write(f"    for(int i=0; i<{NUM_FEATURES}; i++) z += x[i] * W[i];\n")
        f.write("    return 1.0f / (1.0f + expf(-z));\n")
        f.write("}\n")
    
    # Print Normalization Constants
    sys.stdout = old_stdout # Temporarily restore stdout
    print("\n" + "="*70)
    print("CRITICAL: Copy the following normalization constants to NeuronActivation.c")
    print("="*70)
    
    print(f"static const float MFCC_MEAN[NUM_FEATURES] = {{")
    print("    ", end="")
    for i in range(NUM_FEATURES):
        print(f"{mean[i]:.4f}f, ", end="")
        if (i + 1) % 5 == 0 and i < NUM_FEATURES - 1:
            print("\n    ", end="")
    print(f"\n}}; // {NUM_FEATURES} elements")
    
    print(f"\nstatic const float MFCC_STD[NUM_FEATURES] = {{")
    print("    ", end="")
    for i in range(NUM_FEATURES):
        safe_std = max(std[i], 1e-6) 
        print(f"{safe_std:.4f}f, ", end="")
        if (i + 1) % 5 == 0 and i < NUM_FEATURES - 1:
            print("\n    ", end="")
    print(f"\n}}; // {NUM_FEATURES} elements")
    print("="*70)
    sys.stdout = StringIO() # Restore capture


# --- MAIN EXECUTION ---
if __name__ == "__main__":
    
    sys.stdout = old_stdout # Temporarily restore stdout for execution output
    
    X_raw, y = prepare_data(DATASET_PATH)
    
    if len(X_raw) == 0:
        print("No valid features extracted. Check paths/files.")
    else:
        print(f"\nTotal samples processed: {len(X_raw)}")
        
        # 4. Split and Normalize
        X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

        # Calculate Mean and StdDev ONLY on the training data
        X_mean = np.mean(X_train_raw, axis=0)
        X_std = np.std(X_train_raw, axis=0)
        
        X_train_norm = (X_train_raw - X_mean) / (X_std + 1e-7)
        X_test_norm = (X_test_raw - X_mean) / (X_std + 1e-7)

        
        # --- CLASS WEIGHT CALCULATION ---
        counts = Counter(y_train)
        total_samples = len(y_train)
        
        # Handle cases where a class might be zero after filtering/splitting
        weight_0 = total_samples / (2 * counts.get(0.0, 1))
        weight_1 = total_samples / (2 * counts.get(1.0, 1))
        
        class_weights = {0: weight_0, 1: weight_1}
        print(f"\nCalculated Class Weights: {class_weights}")
        print(f"Target (1.0) samples: {counts.get(1.0, 0)}, Weight: {weight_1:.2f}")
        print(f"Other (0.0) samples: {counts.get(0.0, 0)}, Weight: {weight_0:.2f}")


        # 5. Training Keras Model
        print(f"\nStarting Keras Training ({len(X_train_norm)} samples) with class weights...")
        
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(NUM_FEATURES,)),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        history = model.fit(
            X_train_norm, y_train, 
            epochs=50, 
            batch_size=32, 
            class_weight=class_weights, 
            validation_data=(X_test_norm, y_test),
            verbose=1
        )
        
        # 6. Export Weights, Bias, and Normalization Constants
        # Restore capture before calling export_c to ensure its prints are routed
        sys.stdout = StringIO() 
        export_c(model, EXPORT_NAME, X_mean, X_std)
        
        # Restore stdout to print the final output
        sys.stdout = old_stdout 
        print("Model configuration and normalization constants successfully exported.")

Total WAV files found: 3000. Filtering for '0' and '1'...
Processed samples after filtering: 600

Total samples processed: 600

Calculated Class Weights: {0: 0.9917355371900827, 1: 1.0084033613445378}
Target (1.0) samples: 238, Weight: 1.01
Other (0.0) samples: 242, Weight: 0.99

Starting Keras Training (480 samples) with class weights...
Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7250 - loss: 0.5610 - val_accuracy: 0.7500 - val_loss: 0.4960
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7437 - loss: 0.5400 - val_accuracy: 0.7500 - val_loss: 0.4769
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7583 - loss: 0.5195 - val_accuracy: 0.7500 - val_loss: 0.4590
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7688 - loss: 0.5007 - val_accuracy: 0.7667 - val_loss: 0.4423
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7750 - loss: 0.4819 - val_accuracy: 0.7833 - val_loss: 0.4266
Epoch 6/50
15/15 ━━━━━━━━━━━

In [16]:


# NOTE: This script assumes 'model' (Keras Sequential) and 
# 'X_test_norm', 'y_test' (NumPy arrays) are available in the current environment.

import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, accuracy_score

# --- 1. EXTRACT DEPLOYMENT CONSTANTS ---

# Extract weights and bias directly from the trained Keras model
W_DEPLOYED, B_DEPLOYED = model.layers[0].get_weights()
W_DEPLOYED = W_DEPLOYED.flatten()
B_DEPLOYED = B_DEPLOYED[0]

# --- 2. LOCAL NEURON IMPLEMENTATION (C Simulation) ---

def neuron_predict_c_sim(features):
    """ 
    Simulates the single neuron prediction function defined in C,
    using the extracted Keras weights (W_DEPLOYED) and bias (B_DEPLOYED).
    """
    
    # Linear combination (W * X + B)
    # This assumes features is already normalized (X_test_norm)
    z = B_DEPLOYED + np.dot(features, W_DEPLOYED)
    
    # Sigmoid Activation
    # Use standard library math functions matching C (math.exp is fine for Python)
    return 1.0 / (1.0 + np.exp(-z))

# --- 3. EXECUTION AND VALIDATION ---

if 'model' not in locals() or 'X_test_norm' not in locals() or 'y_test' not in locals():
    print("❌ ERROR: Required variables (model, X_test_norm, y_test) not found.")
    print("Ensure this code runs immediately after the Keras training cell.")
    exit()

print("\n--- Local C Implementation Verification ---")

# 3a. Predict using the C-simulated function
probabilities_c_sim = neuron_predict_c_sim(X_test_norm)

# 3b. Predict using the Keras model (baseline comparison)
probabilities_keras = model.predict(X_test_norm, verbose=0).flatten()

# Convert probabilities to binary prediction (0 or 1)
predictions_c_sim = (probabilities_c_sim > 0.5).astype(int)
y_test_bin = y_test.astype(int)

# Calculate metrics
acc_c_sim = accuracy_score(y_test_bin, predictions_c_sim)
cm = confusion_matrix(y_test_bin, predictions_c_sim)

print("-" * 50)
print(f"Total Test Samples: {len(X_test_norm)}")
print(f"C Simulation Accuracy: {acc_c_sim * 100:.4f}%")
print(f"Keras Validation Accuracy: {accuracy_score(y_test_bin, (probabilities_keras > 0.5).astype(int)) * 100:.4f}%")
print("-" * 50)
print("Confusion Matrix (C Sim):")
print(f"| TN: {cm[0, 0]} | FP: {cm[0, 1]} |")
print(f"| FN: {cm[1, 0]} | TP: {cm[1, 1]} |")
print("-" * 50)

# Verify if C constants perfectly match the training weights (optional check)
if np.allclose(W_DEPLOYED, model.layers[0].get_weights()[0].flatten()) and np.allclose(B_DEPLOYED, model.layers[0].get_weights()[1][0]):
    print("✅ C Constants (W, B) extracted match the Keras model perfectly.")
else:
    print("⚠️ WARNING: C Constants (W, B) extraction mismatch.")


--- Local C Implementation Verification ---
--------------------------------------------------
Total Test Samples: 120
C Simulation Accuracy: 99.1667%
Keras Validation Accuracy: 99.1667%
--------------------------------------------------
Confusion Matrix (C Sim):
| TN: 57 | FP: 1 |
| FN: 0 | TP: 62 |
--------------------------------------------------
✅ C Constants (W, B) extracted match the Keras model perfectly.
